# 2. Train and explain the full-cohort EEG models
Attach the saved output of notebook 01 using Add Input. Enable a GPU and Internet. The CNN uses one CUDA device; classical methods use CPU. This notebook does not use the eight-subject pilot by default.

In [ ]:
from pathlib import Path
import subprocess, sys, shutil

REPO = Path('/kaggle/working/TTKHTKUD_26-27')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/trungnd06/TTKHTKUD_26-27.git', str(REPO)], check=True)
SOURCE = REPO / 'kaggle'
assert (SOURCE / 'prepare_data.py').is_file(), 'This clone does not contain kaggle/. Use the updated repository.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(SOURCE / 'requirements.txt')], check=True)
print('Code:', SOURCE)


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Enable a Kaggle GPU before training, or deliberately select DEVICE=cpu and remove this guard.')


In [ ]:
# Fill DATA_PATH if automatic discovery finds more than one file.
DATA_PATH = None  # Example: '/kaggle/input/my-full-eeg/processed/epochs_full.npz'
RESUME_RESULTS = None  # Optional folder of previously saved results, including cached_folds/ and models/.
RESULTS = Path('/kaggle/working/nencki_full/results')
DEVICE = 'cuda'
EPOCHS = 50
FOLDS = 5
SEED = 42

if DATA_PATH is None:
    candidates = sorted(Path('/kaggle/input').rglob('epochs_full.npz'))
    local = Path('/kaggle/working/nencki_full/processed/epochs_full.npz')
    if local.is_file():
        candidates.append(local)
    if len(candidates) != 1:
        raise RuntimeError(f'Found {len(candidates)} prepared datasets. Attach notebook 01 output or set DATA_PATH explicitly. {candidates}')
    DATA_PATH = candidates[0]
DATA_PATH = Path(DATA_PATH)
assert (DATA_PATH.parent / 'preparation_summary.json').is_file()
if RESUME_RESULTS is not None:
    source = Path(RESUME_RESULTS).resolve()
    assert source.is_dir() and source != RESULTS.resolve()
    shutil.copytree(source, RESULTS, dirs_exist_ok=True)
print('Data:', DATA_PATH)


In [ ]:
subprocess.run([sys.executable, str(SOURCE / 'train.py'), '--data', str(DATA_PATH),
                '--output', str(RESULTS), '--device', DEVICE, '--epochs', str(EPOCHS),
                '--folds', str(FOLDS), '--seed', str(SEED)], check=True)


In [ ]:
subprocess.run([sys.executable, str(SOURCE / 'verify.py'), '--data', str(DATA_PATH),
                '--output', str(RESULTS)], check=True)
import pandas as pd
display(pd.read_csv(RESULTS / 'summary.csv', header=[0, 1], index_col=0))
print((RESULTS / 'RESULTS_EN.md').read_text())


In [ ]:
from IPython.display import Image, display
for name in ['comparison.png', 'erp.png', 'attribution.png', 'occlusion.png']:
    display(Image(filename=str(RESULTS / name)))


In [ ]:
# Save Version with outputs, or download this results archive.
archive = shutil.make_archive('/kaggle/working/nencki_full_results', 'zip', root_dir=RESULTS)
print(archive)


## Interpret the measured results
Use the actual retained subject count and QC exclusions in the report. One base seed and fold SD do not establish statistical or universal superiority. Attributions are model sensitivity, not neural-source localization. The parent project report remains the eight-subject pilot until the group updates it using these measured results.